In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from topostats.io import LoadScans
from topostats.filters import Filters
from topostats.plottingfuncs import Colormap
colormap = Colormap()
cmap = colormap.get_cmap()

In [ ]:
def plot(image: np.ndarray):
    plt.imshow(image, cmap=cmap)
    plt.show()

In [ ]:
heightmap = np.zeros((100, 100))
for y in range(heightmap.shape[0]):
    for x in range(heightmap.shape[1]):
        heightmap[y, x] += 3.0 + 0.5*(x-50)*(y-50)
# heightmap += np.random.random((heightmap.shape[0], heightmap.shape[1]))

plt.imshow(heightmap, cmap="twilight")
plt.show()


def model_func(x, y, a, b, c, d):
    return a + b * x * y - c*x - d*y

# your data
xdata, ydata = np.meshgrid(np.arange(heightmap.shape[1]), np.arange(heightmap.shape[0]))
zdata = heightmap.ravel()

print('xdata')
print(xdata)
print('ydata')
print(ydata)
print('zdata')
print(zdata)
print()

xy_data_stacked = np.vstack((xdata.ravel(), ydata.ravel()))
print("xy data stacked:")
print(xy_data_stacked)
print()

# fit the model to your data
popt, pcov = curve_fit(lambda x, a, b, c, d: model_func(x[0], x[1], a, b, c, d), xy_data_stacked, zdata)

# popt contains the optimal values for the parameters a and b
a, b, c, d = popt

print("---- results ----")
print(f"a: {a}, b:{b} c:{c}, d:{d}")
print()

print("--- cov ---")
print(pcov)
plt.imshow(pcov)
plt.show()

# now you can use the fitted model to make predictions
z_pred = model_func(xdata, ydata, a, b, c, d)

plt.imshow(z_pred, cmap='twilight')

In [ ]:
file = Path()
loadscans = LoadScans([file], channel="Height")
loadscans.get_data()
image = loadscans.image
pixel_to_nm_scaling = loadscans.pixel_to_nm_scaling

plot(image)

filters = Filters(
    image=image,
    filename="image",
    pixel_to_nm_scaling=pixel_to_nm_scaling
)

image = filters.median_flatten(image)
plot(image)
image = filters.remove_tilt(image)
plot(image)


heightmap = image

# Fit dependant polynomial
def model_func(x, y, a, b, c, d):
    return a + b * x * y - c*x - d*y

xdata, ydata = np.meshgrid(np.arange(heightmap.shape[1]), np.arange(heightmap.shape[0]))
zdata = heightmap.ravel()

print('xdata')
print(xdata)
print('ydata')
print(ydata)
print('zdata')
print(zdata)
print()

xy_data_stacked = np.vstack((xdata.ravel(), ydata.ravel()))
print("xy data stacked:")
print(xy_data_stacked)
print()

# fit the model to your data
popt, pcov = curve_fit(lambda x, a, b, c, d: model_func(x[0], x[1], a, b, c, d), xy_data_stacked, zdata)

# popt contains the optimal values for the parameters a and b
a, b, c, d = popt

print("---- results ----")
print(f"a: {a}, b:{b} c:{c}, d:{d}")
print()

print("--- cov ---")
print(pcov)
plt.imshow(pcov)
plt.show()

# now you can use the fitted model to make predictions
z_pred = model_func(xdata, ydata, a, b, c, d)

plt.imshow(z_pred, cmap='twilight')

image -= z_pred

plt.imshow(image, vmin=-20, vmax=10, cmap=cmap)

image = filters.median_flatten(image)

plt.imshow(image, vmin=-20, vmax=10, cmap=cmap)